# Image Layers Pipeline

Añade `dish_image_path` (join con Epicurious) e `ingredient_images` (catálogo Fruits-360 + fasihcs) al dataset de recetas.

Outputs: `df_clean_final.parquet`, `ingredient_catalog.json`, `ingredient_coverage.csv`

In [16]:
# ==========================================
# 📦 INSTALLS
# ==========================================
!pip install kagglehub rapidfuzz pyarrow -q
print("kagglehub + rapidfuzz + pyarrow listos")

kagglehub + rapidfuzz + pyarrow listos


In [17]:
# ==========================================
# 📦 IMPORTS
# ==========================================
import kagglehub
import pandas as pd
import json
import os
import re
from pathlib import Path
from collections import defaultdict
from IPython.display import display

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tiff'}
print("Imports OK")

Imports OK


In [18]:
# ==========================================
# 🔧 FUNCIONES AUXILIARES
# ==========================================

def get_image_files(directory):
    # Sorted list of image files in a folder (non-recursive)
    p = Path(directory)
    if not p.exists() or not p.is_dir():
        return []
    return sorted(
        f for f in p.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS
    )


def count_classes(root_dir):
    # {class_name: n_images} for every immediate subfolder that contains images
    root = Path(root_dir)
    result = {}
    if not root.exists():
        return result
    for d in sorted(root.iterdir()):
        if d.is_dir():
            n = sum(
                1 for f in d.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTS
            )
            if n > 0:
                result[d.name] = n
    return dict(sorted(result.items(), key=lambda x: -x[1]))


def print_tree(root, depth=2, _d=0, _pfx=""):
    # Print a directory tree up to `depth` levels (max 25 items per level)
    root = Path(root)
    if _d > depth:
        return
    try:
        all_items = sorted(root.iterdir())
    except PermissionError:
        return
    items = all_items[:25]
    for i, item in enumerate(items):
        conn = "└── " if i == len(items) - 1 else "├── "
        if item.is_dir():
            try:
                sub = len(list(item.iterdir()))
            except Exception:
                sub = "?"
            print(f"{_pfx}{conn}{item.name}/  [{sub} items]")
            if _d < depth:
                nxt = _pfx + ("    " if i == len(items) - 1 else "│   ")
                print_tree(item, depth, _d + 1, nxt)
        else:
            sz = item.stat().st_size
            sz_s = f"{sz / 1024:.0f} KB" if sz < 1_048_576 else f"{sz / 1_048_576:.1f} MB"
            print(f"{_pfx}{conn}{item.name}  ({sz_s})")
    overflow = len(all_items) - len(items)
    if overflow > 0:
        print(f"{_pfx}    ... +{overflow} items más")


def clean_key(name):
    # Normalize folder name → catalog key: lowercase, no digits, no hyphens/underscores
    name = name.lower()
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'[-_]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name


print("Helper functions cargadas")

Helper functions cargadas


---
## Dataset 1 — 64k Recipes
`wafaaelhusseini/extended-recipes-dataset-64k-dishes`

In [19]:
# ==========================================
# DATASET 1: 64k Recipes
# ==========================================
print("=" * 65)
print("DATASET 1: wafaaelhusseini/extended-recipes-dataset-64k-dishes")
print("=" * 65)

path_64k = kagglehub.dataset_download(
    "wafaaelhusseini/extended-recipes-dataset-64k-dishes"
)
print(f"\nPath de descarga: {path_64k}")

print("\nEstructura:")
print_tree(path_64k, depth=1)

# Localizar CSV
csv_files = list(Path(path_64k).glob("**/*.csv"))
print(f"\nCSV encontrados: {[f.name for f in csv_files]}")

df = pd.read_csv(csv_files[0])

print(f"\n--- Shape ---")
print(df.shape)

print(f"\n--- Columnas disponibles ({len(df.columns)}) ---")
for col in df.columns:
    dtype = str(df[col].dtype)
    print(f"  {col:<40} {dtype}")

print("\n--- Sample 3 filas ---")
display(df.head(3))

DATASET 1: wafaaelhusseini/extended-recipes-dataset-64k-dishes

Path de descarga: /home/ramon/.cache/kagglehub/datasets/wafaaelhusseini/extended-recipes-dataset-64k-dishes/versions/1

Estructura:
├── recipes_extended.csv  (285.0 MB)
└── recipes_extended.json  (347.3 MB)

CSV encontrados: ['recipes_extended.csv']

--- Shape ---
(62126, 38)

--- Columnas disponibles (38) ---
  recipe_title                             str
  category                                 str
  subcategory                              str
  description                              str
  ingredients                              str
  directions                               str
  num_ingredients                          int64
  num_steps                                int64
  ingredient_text                          str
  directions_text                          str
  combined_text                            str
  ingredients_raw                          str
  directions_raw                           str
  ingredi

,recipe_title,category,subcategory,description,ingredients,directions,num_ingredients,num_steps,ingredient_text,directions_text,...,is_halal,is_kosher,is_nut_free,is_dairy_free,is_gluten_free,dietary_profile,healthiness_score,health_flags,main_ingredient,health_level
0,Air Fryer Potato Slices with Dipping Sauce,Air Fryer Recipes,Air Fryer Recipes,"These air fryer potato slices, served with a b...","[""3/4 cup ketchup"", ""1/2 cup beer"", ""1 tablesp...","[""Combine ketchup, beer, Worcestershire sauce,...",9,5,ketchup beer worcestershire sauce onion powder...,"combine ketchup, beer, worcestershire sauce, o...",...,False,True,True,True,True,"[""vegan"", ""gluten_free"", ""nut_free"", ""kosher""]",80,"[""plant_based"", ""healthy_fats"", ""fried""]",unknown,healthy
1,Gochujang Pork Belly Bites,Air Fryer Recipes,Air Fryer Recipes,These gochujang pork belly bites are sweet and...,"[""1 pound pork belly"", ""1/4 cup gochujang"", ""2...","[""Preheat an air fryer to 400 degrees F (200 d...",5,4,pound pork belly gochujang soy sauce honey gro...,preheat an air fryer to 400 degrees f (200 deg...,...,False,False,True,True,True,"[""gluten_free"", ""nut_free""]",64,"[""sugary""]",red_meat,moderate
2,3-Ingredient Air Fryer Everything Bagel Chicke...,Air Fryer Recipes,Air Fryer Recipes,These 3-ingredient air fryer everything bagel ...,"[""1 ¼ pounds chicken tenders"", ""1 tablespoon o...","[""Gather all ingredients. Preheat an air fryer...",3,4,¼ pounds chicken tenders olive oil everything ...,gather all ingredients. preheat an air fryer t...,...,True,True,True,True,True,"[""gluten_free"", ""nut_free"", ""halal"", ""kosher""]",68,"[""healthy_fats"", ""fried""]",poultry,moderate


---
## Dataset 2 — Epicurious
`pes12017000148/food-ingredients-and-recipe-dataset-with-images`

In [20]:
# ==========================================
# DATASET 2: Epicurious — dish images
# ==========================================
print("=" * 65)
print("DATASET 2: food-ingredients-and-recipe-dataset-with-images")
print("=" * 65)

path_epi = kagglehub.dataset_download(
    "pes12017000148/food-ingredients-and-recipe-dataset-with-images"
)
print(f"\nPath de descarga: {path_epi}")

print("\nEstructura de directorios:")
print_tree(path_epi, depth=2)

# ── CSV ──────────────────────────────────────────────────────────────────────
csv_files_epi = list(Path(path_epi).glob("**/*.csv"))
if csv_files_epi:
    df_epi = pd.read_csv(csv_files_epi[0])
    print(f"\n--- Shape ---")
    print(df_epi.shape)
    print(f"\n--- Columnas disponibles ({len(df_epi.columns)}) ---")
    for col in df_epi.columns:
        dtype = str(df_epi[col].dtype)
        print(f"  {col:<40} {dtype}")
    print("\n--- Sample 3 filas ---")
    display(df_epi.head(3))
else:
    print("\nNo se encontró CSV en el dataset")

# ── Imagen del platillo: detectar estructura ──────────────────────────────────
print("\n--- Clases/Carpetas de imágenes del platillo ---")
dirs_with_imgs = {}
for d in sorted(Path(path_epi).rglob("*")):
    if d.is_dir():
        n = sum(
            1 for f in d.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS
        )
        if n > 0:
            dirs_with_imgs[d] = n

if dirs_with_imgs:
    parents = {d.parent for d in dirs_with_imgs}
    if len(parents) == 1:
        # Carpetas de clase bajo un mismo directorio raíz
        classes_epi = {d.name: n for d, n in dirs_with_imgs.items()}
        classes_epi = dict(sorted(classes_epi.items(), key=lambda x: -x[1]))
        print(f"Total clases: {len(classes_epi)}")
        print(f"Total imágenes: {sum(classes_epi.values())}")
        print(f"\nTop 20 clases:")
        for cls, n in list(classes_epi.items())[:20]:
            print(f"  {cls:<40} {n} imágenes")
        if len(classes_epi) > 20:
            print(f"  ... y {len(classes_epi) - 20} más")
    else:
        print(f"Imágenes distribuidas en {len(dirs_with_imgs)} carpetas:")
        for d, n in sorted(dirs_with_imgs.items(), key=lambda x: -x[1])[:20]:
            rel = d.relative_to(path_epi)
            print(f"  {str(rel):<50} {n} imágenes")
else:
    all_imgs = (
        list(Path(path_epi).rglob("*.jpg"))
        + list(Path(path_epi).rglob("*.jpeg"))
        + list(Path(path_epi).rglob("*.png"))
    )
    print(f"Imágenes flat (sin subcarpetas de clase): {len(all_imgs)}")
    if all_imgs:
        print(f"Ejemplo: {all_imgs[0]}")

DATASET 2: food-ingredients-and-recipe-dataset-with-images

Path de descarga: /home/ramon/.cache/kagglehub/datasets/pes12017000148/food-ingredients-and-recipe-dataset-with-images/versions/1

Estructura de directorios:
├── Food Images/  [1 items]
│   └── Food Images/  [13582 items]
│       ├── -bloody-mary-tomato-toast-with-celery-and-horseradish-56389813.jpg  (17 KB)
│       ├── -burnt-carrots-and-parsnips-56390131.jpg  (19 KB)
│       ├── -candy-corn-frozen-citrus-cream-pops-368770.jpg  (12 KB)
│       ├── -candy-corn-pumpkin-blondies-51254510.jpg  (10 KB)
│       ├── -carbonnade-a-la-flamande-short-ribs-358557.jpg  (11 KB)
│       ├── -chickpea-barley-and-feta-salad-51239040.jpg  (20 KB)
│       ├── -chickpea-pancakes-with-leeks-squash-and-yogurt-51260630.jpg  (17 KB)
│       ├── -cod-with-mussels-chorizo-fried-croutons-and-saffron-mayonnaise-355204.jpg  (12 KB)
│       ├── -em-ba-em-s-ultimate-lobster-rolls-51169080.jpg  (7 KB)
│       ├── -em-gourmet-live-em-s-first-birthday-cake-3

,Unnamed: 0,Title,Ingredients,Instructions,Image_Name,Cleaned_Ingredients
0,0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...","Pat chicken dry with paper towels, season all ...",miso-butter-roast-chicken-acorn-squash-panzanella,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher..."
1,1,Crispy Salt and Pepper Potatoes,"['2 large egg whites', '1 pound new potatoes (...",Preheat oven to 400°F and line a rimmed baking...,crispy-salt-and-pepper-potatoes-dan-kluger,"['2 large egg whites', '1 pound new potatoes (..."
2,2,Thanksgiving Mac and Cheese,"['1 cup evaporated milk', '1 cup whole milk', ...",Place a rack in middle of oven; preheat to 400...,thanksgiving-mac-and-cheese-erick-williams,"['1 cup evaporated milk', '1 cup whole milk', ..."



--- Clases/Carpetas de imágenes del platillo ---
Total clases: 1
Total imágenes: 13582

Top 20 clases:
  Food Images                              13582 imágenes


---
## Dataset 3 — Fruits-360
`moltean/fruits`

In [21]:
# ==========================================
# DATASET 3: Fruits-360
# ==========================================
print("=" * 65)
print("DATASET 3: moltean/fruits  (Fruits-360)")
print("=" * 65)

path_fruits = kagglehub.dataset_download("moltean/fruits")
print(f"\nPath de descarga: {path_fruits}")

print("\nEstructura (primeros 2 niveles):")
print_tree(path_fruits, depth=2)

# Localizar splits Training / Test
training_dirs = list(Path(path_fruits).rglob("Training"))
test_dirs      = list(Path(path_fruits).rglob("Test"))

print(f"\nCarpetas Training: {[str(d.relative_to(path_fruits)) for d in training_dirs]}")
print(f"Carpetas Test:     {[str(d.relative_to(path_fruits)) for d in test_dirs]}")

# Usar Training para el catálogo
fruits_train_dir = training_dirs[0] if training_dirs else None

if fruits_train_dir:
    classes_fruits = count_classes(fruits_train_dir)
    total_imgs_fruits = sum(classes_fruits.values())

    print(f"\n--- Shape (split Training) ---")
    print(f"  Clases: {len(classes_fruits)}")
    print(f"  Total imágenes: {total_imgs_fruits}")

    print(f"\n--- Clases disponibles con # imágenes ---")
    for cls, n in list(classes_fruits.items()):
        print(f"  {cls:<40} {n}")

    print(f"\n--- Sample 3 clases (ruta de imagen representativa) ---")
    for cls in list(classes_fruits.keys())[:3]:
        imgs = get_image_files(fruits_train_dir / cls)
        print(f"  Clase : '{cls}'")
        print(f"  Ruta  : {imgs[0] if imgs else 'N/A'}")
        print()
else:
    print("No se encontró carpeta Training")
    fruits_train_dir = None
    classes_fruits = {}

# CSV (si existe)
csv_fruits = list(Path(path_fruits).glob("**/*.csv"))
if csv_fruits:
    df_fruits_meta = pd.read_csv(csv_fruits[0])
    print(f"\n--- CSV metadata ---")
    print(f"Shape: {df_fruits_meta.shape}")
    print(f"Columnas: {df_fruits_meta.columns.tolist()}")
    display(df_fruits_meta.head(3))

DATASET 3: moltean/fruits  (Fruits-360)

Path de descarga: /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89

Estructura (primeros 2 niveles):
├── fruits-360_100x100/  [1 items]
│   └── fruits-360/  [4 items]
│       ├── LICENSE  (1 KB)
│       ├── README.md  (7 KB)
│       ├── Test/  [260 items]
│       └── Training/  [260 items]
├── fruits-360_3-body-problem/  [1 items]
│   └── fruits-360-3-body-problem/  [4 items]
│       ├── LICENSE  (1 KB)
│       ├── README.md  (2 KB)
│       ├── Test/  [3 items]
│       └── Training/  [3 items]
├── fruits-360_meta/  [1 items]
│   └── fruits-360-meta/  [5 items]
│       ├── LICENSE  (1 KB)
│       ├── Meta/  [253 items]
│       ├── Papers/  [1 items]
│       ├── README.md  (2 KB)
│       └── dictionary.txt  (1 KB)
├── fruits-360_multi/  [3 items]
│   ├── LICENSE  (1 KB)
│   ├── README.md  (1 KB)
│   └── test-multiple_fruits/  [217 items]
│       ├── Banana(lady_finger)_1.jpg  (1.3 MB)
│       ├── Banana(lady_finger)_2.jpg  (1.5 MB)

---
## Dataset 4 — Recipe Ingredients
`fasihcs/recipe-ingredients-image-dataset`

In [22]:
# ==========================================
# DATASET 4: Recipe Ingredients Image Dataset
# ==========================================
print("=" * 65)
print("DATASET 4: fasihcs/recipe-ingredients-image-dataset")
print("=" * 65)

path_ingr = kagglehub.dataset_download("fasihcs/recipe-ingredients-image-dataset")
print(f"\nPath de descarga: {path_ingr}")

print("\nEstructura (primeros 3 niveles):")
print_tree(path_ingr, depth=3)

# Recopilar todos los directorios que contienen imágenes
classes_ingr_by_dir = {}
for d in sorted(Path(path_ingr).rglob("*")):
    if d.is_dir():
        n = sum(
            1 for f in d.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS
        )
        if n > 0:
            classes_ingr_by_dir[d] = n

total_ingr_imgs = sum(classes_ingr_by_dir.values())

print(f"\n--- Shape ---")
print(f"  Directorios con imágenes: {len(classes_ingr_by_dir)}")
print(f"  Total imágenes: {total_ingr_imgs}")

print(f"\n--- Clases/Carpetas disponibles con # imágenes ---")
for d, n in sorted(classes_ingr_by_dir.items(), key=lambda x: -x[1]):
    rel = d.relative_to(path_ingr)
    print(f"  {str(rel):<50} {n}")

print(f"\n--- Sample 3 clases (ruta de imagen representativa) ---")
for d, n in list(sorted(classes_ingr_by_dir.items(), key=lambda x: -x[1]))[:3]:
    imgs = get_image_files(d)
    rel = d.relative_to(path_ingr)
    print(f"  Clase : '{d.name}'  (ruta relativa: {rel})")
    print(f"  Imgs  : {n}  |  Ejemplo: {imgs[0].name if imgs else 'N/A'}")
    print()

# CSV
csv_ingr = list(Path(path_ingr).glob("**/*.csv"))
if csv_ingr:
    df_ingr = pd.read_csv(csv_ingr[0])
    print(f"\n--- CSV metadata ---")
    print(f"Shape: {df_ingr.shape}")
    print(f"Columnas: {df_ingr.columns.tolist()}")
    display(df_ingr.head(3))

DATASET 4: fasihcs/recipe-ingredients-image-dataset

Path de descarga: /home/ramon/.cache/kagglehub/datasets/fasihcs/recipe-ingredients-image-dataset/versions/1

Estructura (primeros 3 niveles):
├── train/  [50 items]
│   ├── Unlabeled/  [16 items]
│   │   ├── Image_54_png.rf.058c46ca8ecf977345726c08b050061e.jpg  (55 KB)
│   │   ├── Image_54_png.rf.4979039e6dc8a7a55d60991dea8724f7.jpg  (59 KB)
│   │   ├── Imageeee-22-_jpg.rf.5cb8b2fda5937d1fb1c4da0ffea0b8f2.jpg  (26 KB)
│   │   ├── Imageeee-22-_jpg.rf.c26e5e5a23110f1ecfa2d5b0d1779018.jpg  (29 KB)
│   │   ├── corn-tortillas_21_jpg.rf.444e1331a930880b2975ee82139e9eea.jpg  (54 KB)
│   │   ├── corn-tortillas_21_jpg.rf.a1ab1584bfa4605dcb9a7a35a34f876a.jpg  (54 KB)
│   │   ├── corn-tortillas_4_jpg.rf.4ae22da325ea80d2fdd787a8617f3d89.jpg  (62 KB)
│   │   ├── corn-tortillas_4_jpg.rf.cfa9d3a1100ef0dad2c2f5b86e618ca2.jpg  (62 KB)
│   │   ├── corn-tortillas_7_jpg.rf.27101c4796bc9a317a53c72301c648c6.jpg  (43 KB)
│   │   ├── corn-tortillas_7_jpg.rf

---
## Construcción del `ingredient_catalog`

`{ "tomato": "/ruta/imagen.jpg", ... }`. Keys: lowercase, sin dígitos ni guiones.
Prioridad: Fruits-360 > fasihcs.

In [23]:
# ==========================================
# 🗂️ CONSTRUIR ingredient_catalog
# ==========================================
print("=" * 65)
print("CONSTRUYENDO ingredient_catalog")
print("=" * 65)

ingredient_catalog   = {}   # { key: "/ruta/absoluta/imagen.jpg" }
ingredient_img_counts = {}  # { key: total_images }  — para estadísticas

# ──────────────────────────────────────────
# PRIORIDAD 1: Fruits-360 Training
# ──────────────────────────────────────────
if fruits_train_dir and fruits_train_dir.exists():
    for class_dir in sorted(fruits_train_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        imgs = get_image_files(class_dir)
        if not imgs:
            continue
        key = clean_key(class_dir.name)
        if not key:
            continue
        if key not in ingredient_catalog:
            ingredient_catalog[key] = str(imgs[0])
        ingredient_img_counts[key] = (
            ingredient_img_counts.get(key, 0) + len(imgs)
        )

n_after_fruits = len(ingredient_catalog)
print(f"Fruits-360 aportó       : {n_after_fruits} ingredientes")

# ──────────────────────────────────────────
# PRIORIDAD 2: Recipe Ingredients Image Dataset
# ──────────────────────────────────────────
for dir_path, n_imgs in sorted(classes_ingr_by_dir.items()):
    key = clean_key(dir_path.name)
    if not key:
        continue
    imgs = get_image_files(dir_path)
    if not imgs:
        continue
    if key not in ingredient_catalog:
        ingredient_catalog[key] = str(imgs[0])
    ingredient_img_counts[key] = (
        ingredient_img_counts.get(key, 0) + n_imgs
    )

n_new_fasihcs = len(ingredient_catalog) - n_after_fruits
print(f"fasihcs aportó          : {n_new_fasihcs} nuevos ingredientes")
print(f"TOTAL en catálogo       : {len(ingredient_catalog)}")

print("\nEjemplos — primeros 15 entries:")
for k, v in list(ingredient_catalog.items())[:15]:
    print(f"  {k:<30} {v}")

CONSTRUYENDO ingredient_catalog
Fruits-360 aportó       : 141 ingredientes
fasihcs aportó          : 42 nuevos ingredientes
TOTAL en catálogo       : 183

Ejemplos — primeros 15 entries:
  almonds                        /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Almonds 1/r0_0_100.jpg
  apple                          /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Apple 10/r0_0_100.jpg
  apple braeburn                 /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Apple Braeburn 1/0_100.jpg
  apple crimson snow             /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Apple Crimson Snow 1/0_100.jpg
  apple golden                   /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Apple Golden 1/0_100.jpg


---
## Guardar `ingredient_catalog.json`

In [24]:
# ==========================================
# 💾 GUARDAR JSON
# ==========================================
output_path = os.path.join(os.getcwd(), "ingredient_catalog.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(ingredient_catalog, f, ensure_ascii=False, indent=2)

file_size_kb = os.path.getsize(output_path) / 1024

print(f"Guardado en    : {output_path}")
print(f"Tamaño         : {file_size_kb:.1f} KB")
print(f"Total entradas : {len(ingredient_catalog)}")

Guardado en    : /home/ramon/DeepLearing_Recipes/ingredient_catalog.json
Tamaño         : 28.1 KB
Total entradas : 183


---
## Estadísticas

In [25]:
# ==========================================
# 📈 ESTADÍSTICAS FINALES
# ==========================================
print("=" * 65)
print("ESTADÍSTICAS FINALES")
print("=" * 65)

print(f"\nTotal de ingredientes en el catálogo : {len(ingredient_catalog)}")

# Top 10 por número de imágenes
print("\nTop 10 clases con más imágenes:")
top10 = sorted(ingredient_img_counts.items(), key=lambda x: -x[1])[:10]
for rank, (key, count) in enumerate(top10, 1):
    img_path = ingredient_catalog.get(key, "N/A")
    print(f"  {rank:2d}. {key:<30} {count:>5} imágenes  →  {img_path}")

# Clases con 1 sola imagen
outliers = sorted(k for k, v in ingredient_img_counts.items() if v == 1)
print(f"\nClases con solo 1 imagen (posibles outliers): {len(outliers)}")
if outliers:
    for k in outliers:
        print(f"  - '{k}'  →  {ingredient_catalog[k]}")
else:
    print("  (ninguna — todas las clases tienen más de 1 imagen)")

ESTADÍSTICAS FINALES

Total de ingredientes en el catálogo : 183

Top 10 clases con más imágenes:
   1. apple                          10167 imágenes  →  /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Apple 10/r0_0_100.jpg
   2. pear                            8432 imágenes  →  /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Pear 1/0_100.jpg
   3. tomato                          6603 imágenes  →  /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Tomato 1/0_100.jpg
   4. cucumber                        6507 imágenes  →  /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Cucumber 1/r0_0_100.jpg
   5. peach                           4311 imágenes  →  /home/ramon/.cache/kagglehub/datasets/moltean/fruits/versions/89/fruits-360_100x100/fruits-360/Training/Peach 1/0_100.jpg
  

---
## Capa 1 — Imagen del platillo

Join `df_clean` × Epicurious sobre `title_norm`. Pasada 1: exact match. Pasada 2: fuzzy (`token_sort_ratio ≥ 88`).

### Normalización de títulos

In [26]:
# ==========================================
# NORMALIZACIÓN DE TÍTULOS
# ==========================================
COOKING_STOPWORDS = {"recipe", "easy", "best", "homemade", "classic", "simple", "quick"}

def normalize_title(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    words = [w for w in text.split() if w not in COOKING_STOPWORDS]
    return " ".join(words).strip()

# ── Verificar / reconstruir df_clean ─────────────────────────────────────────
try:
    _ = df_clean
    print(f"df_clean en memoria  : {df_clean.shape}")
except NameError:
    print("df_clean no encontrado — reconstruyendo desde el dataset 64k...")
    csv_64k = list(Path(path_64k).glob("**/*.csv"))
    df_clean = pd.read_csv(csv_64k[0])
    print(f"df_clean cargado     : {df_clean.shape}")

title_col = "recipe_title" if "recipe_title" in df_clean.columns else "title"
print(f"Columna de título    : '{title_col}'")

df_clean["title_norm"] = df_clean[title_col].apply(normalize_title)

# ── Verificar / cargar df_epi ─────────────────────────────────────────────────
try:
    _ = df_epi
except NameError:
    print("df_epi no encontrado — cargando Epicurious...")
    csv_epi = list(Path(path_epi).glob("**/*.csv"))
    df_epi = pd.read_csv(csv_epi[0])

# Rutas absolutas de imagen: {path_epi}/Food Images/Food Images/{Image_Name}.jpg
img_dir_epi = Path(path_epi) / "Food Images" / "Food Images"
df_epi["dish_image_path"] = df_epi["Image_Name"].apply(
    lambda name: str(img_dir_epi / f"{name}.jpg") if pd.notna(name) else None
)
df_epi["title_norm"] = df_epi["Title"].apply(normalize_title)

print(f"df_epi               : {df_epi.shape}")
print(f"img_dir_epi          : {img_dir_epi}")
print(f"Imágenes disponibles : {len(list(img_dir_epi.glob('*.jpg'))):,}")

print("\nEjemplos title_norm — df_clean:")
print(df_clean[[title_col, "title_norm"]].head(3).to_string(index=False))
print("\nEjemplos title_norm — Epicurious:")
print(df_epi[["Title", "title_norm", "dish_image_path"]].head(3).to_string(index=False))

df_clean en memoria  : (62126, 40)
Columna de título    : 'recipe_title'
df_epi               : (13501, 8)
img_dir_epi          : /home/ramon/.cache/kagglehub/datasets/pes12017000148/food-ingredients-and-recipe-dataset-with-images/versions/1/Food Images/Food Images
Imágenes disponibles : 13,582

Ejemplos title_norm — df_clean:
                                          recipe_title                                            title_norm
            Air Fryer Potato Slices with Dipping Sauce            air fryer potato slices with dipping sauce
                            Gochujang Pork Belly Bites                            gochujang pork belly bites
3-Ingredient Air Fryer Everything Bagel Chicken Strips 3ingredient air fryer everything bagel chicken strips

Ejemplos title_norm — Epicurious:
                                                 Title                                            title_norm                                                                                            

### Pasada 1: Exact Match

In [27]:
# ==========================================
# PASADA 1: EXACT MATCH
# ==========================================
# Epicurious puede tener títulos duplicados — keep first occurrence
epi_lookup = (
    df_epi
    .drop_duplicates(subset=["title_norm"])
    [["title_norm", "dish_image_path"]]
    .copy()
)
print(f"Epicurious — títulos únicos normalizados : {len(epi_lookup):,}")

# Drop columna previa si la hubiera (re-ejecución segura)
df_clean = df_clean.drop(columns=["dish_image_path"], errors="ignore")

df_clean = df_clean.merge(epi_lookup, on="title_norm", how="left")

n_exact = int(df_clean["dish_image_path"].notna().sum())
n_total = len(df_clean)

print(f"\nPasada 1 — exact match:")
print(f"  Con imagen  : {n_exact:>7,}  ({n_exact / n_total:.1%})")
print(f"  Sin imagen  : {n_total - n_exact:>7,}  ({(n_total - n_exact) / n_total:.1%})")

Epicurious — títulos únicos normalizados : 13,264

Pasada 1 — exact match:
  Con imagen  :   1,969  (3.2%)
  Sin imagen  :  60,157  (96.8%)


### Pasada 2: Fuzzy Match

`fuzz.token_sort_ratio ≥ 88`. Batches de 500 con `process.cdist`.

In [28]:
# ==========================================
# PASADA 2: FUZZY MATCH
# ==========================================
from rapidfuzz import process, fuzz
import numpy as np

THRESHOLD  = 88
BATCH_SIZE = 500

unmatched_mask   = df_clean["dish_image_path"].isna()
unmatched_titles = df_clean.loc[unmatched_mask, "title_norm"].tolist()
unmatched_idx    = df_clean.index[unmatched_mask].tolist()

epi_norms = epi_lookup["title_norm"].tolist()
epi_img   = dict(zip(epi_lookup["title_norm"], epi_lookup["dish_image_path"]))

n_unmatched = len(unmatched_titles)
print(f"Pasada 2 — fuzzy match (token_sort_ratio ≥ {THRESHOLD}):")
print(f"  Títulos a resolver : {n_unmatched:,}")

if n_unmatched == 0:
    print("  → Todos resueltos en Pasada 1, no se necesita fuzzy match.")
else:
    n_batches    = (n_unmatched + BATCH_SIZE - 1) // BATCH_SIZE
    fuzzy_results = [None] * n_unmatched

    for b in range(n_batches):
        start = b * BATCH_SIZE
        batch = unmatched_titles[start : start + BATCH_SIZE]

        # cdist devuelve matriz (len(batch) × len(epi_norms)) con scores float32
        scores    = process.cdist(batch, epi_norms,
                                  scorer=fuzz.token_sort_ratio,
                                  dtype=np.float32)
        best_idxs = np.argmax(scores, axis=1)
        best_scrs = scores[np.arange(len(batch)), best_idxs]

        for i, (bi, bs) in enumerate(zip(best_idxs, best_scrs)):
            if float(bs) >= THRESHOLD:
                fuzzy_results[start + i] = epi_img.get(epi_norms[int(bi)])

        # Progreso cada 20 lotes
        if (b + 1) % 20 == 0 or b == n_batches - 1:
            done    = start + len(batch)
            matched = sum(1 for r in fuzzy_results[:done] if r is not None)
            print(f"  {done:>6,}/{n_unmatched:,}  "
                  f"({done / n_unmatched:.0%})  — fuzzy matches: {matched:,}")

    # Asignar resultados al DataFrame
    for idx, path in zip(unmatched_idx, fuzzy_results):
        if path is not None:
            df_clean.at[idx, "dish_image_path"] = path

    n_fuzzy = sum(1 for r in fuzzy_results if r is not None)
    n_final = int(df_clean["dish_image_path"].notna().sum())
    print(f"\n  Nuevos matches fuzzy : {n_fuzzy:,}")
    print(f"  Total con imagen     : {n_final:,}  ({n_final / n_total:.1%})")

Pasada 2 — fuzzy match (token_sort_ratio ≥ 88):
  Títulos a resolver : 60,157
  10,000/60,157  (17%)  — fuzzy matches: 158
  20,000/60,157  (33%)  — fuzzy matches: 467
  30,000/60,157  (50%)  — fuzzy matches: 710
  40,000/60,157  (66%)  — fuzzy matches: 925
  50,000/60,157  (83%)  — fuzzy matches: 1,161
  60,000/60,157  (100%)  — fuzzy matches: 1,388
  60,157/60,157  (100%)  — fuzzy matches: 1,394

  Nuevos matches fuzzy : 1,394
  Total con imagen     : 3,363  (5.4%)


### Cobertura — Capa 1

In [29]:
# ==========================================
# REPORTE DE COBERTURA
# ==========================================
print("=" * 65)
print("REPORTE DE COBERTURA — dish_image_path")
print("=" * 65)

n_total   = len(df_clean)
n_with    = int(df_clean["dish_image_path"].notna().sum())
n_without = n_total - n_with

print(f"\nTotal recetas    : {n_total:,}")
print(f"Con imagen       : {n_with:,}  ({n_with / n_total:.1%})")
print(f"Sin imagen (NaN) : {n_without:,}  ({n_without / n_total:.1%})")

# Distribución por categoría
cat_col = next(
    (c for c in ["category", "category_name", "cuisine"] if c in df_clean.columns),
    None
)
if cat_col:
    cat_stats = (
        df_clean
        .groupby(cat_col, observed=True)["dish_image_path"]
        .agg(total="count", con_imagen=lambda x: x.notna().sum())
        .assign(cobertura=lambda d: (d["con_imagen"] / d["total"]).map("{:.1%}".format))
        .sort_values("con_imagen", ascending=False)
    )
    print(f"\nDistribución por '{cat_col}':")
    display(cat_stats)
else:
    print("\n(columna de categoría no encontrada en df_clean)")

# Muestra de recetas con imagen
title_col = "recipe_title" if "recipe_title" in df_clean.columns else "title"
print("\nMuestra — 5 recetas con imagen asignada:")
sample_cols = [title_col, "title_norm", "dish_image_path"]
display(df_clean[df_clean["dish_image_path"].notna()][sample_cols].head(5))

REPORTE DE COBERTURA — dish_image_path

Total recetas    : 62,126
Con imagen       : 3,363  (5.4%)
Sin imagen (NaN) : 58,763  (94.6%)

Distribución por 'category':


,total,con_imagen,cobertura
category,,,
Main Dishes,185,185,100.0%
Cakes,144,144,100.0%
Healthy Recipes,134,134,100.0%
Cookies,133,133,100.0%
Pies,107,107,100.0%
...,...,...,...
Pasties,0,0,nan%
Pasta Carbonara,0,0,nan%
Pavlovas,0,0,nan%



Muestra — 5 recetas con imagen asignada:


,recipe_title,title_norm,dish_image_path
212,Crispy Air Fryer Chickpeas,crispy air fryer chickpeas,/home/ramon/.cache/kagglehub/datasets/pes12017...
282,Tomato Butter,tomato butter,/home/ramon/.cache/kagglehub/datasets/pes12017...
372,Beet Pickled Deviled Eggs,beet pickled deviled eggs,/home/ramon/.cache/kagglehub/datasets/pes12017...
374,Truffle Deviled Eggs,truffle deviled eggs,/home/ramon/.cache/kagglehub/datasets/pes12017...
521,Homemade Pistachio Cream,pistachio cream,/home/ramon/.cache/kagglehub/datasets/pes12017...


### Guardar `df_clean_with_dish_images.parquet`

In [30]:
# ==========================================
# GUARDAR PARQUET
# ==========================================
output_path = os.path.join(os.getcwd(), "df_clean_with_dish_images.parquet")

df_clean.to_parquet(output_path, index=False, engine="pyarrow")

file_size_mb = os.path.getsize(output_path) / 1_048_576
print(f"Guardado en  : {output_path}")
print(f"Tamaño       : {file_size_mb:.1f} MB")
print(f"Shape        : {df_clean.shape}")

print(f"\nColumnas del DataFrame final:")
for col in df_clean.columns:
    n_null  = int(df_clean[col].isna().sum())
    pct_null = n_null / len(df_clean)
    flag = " ◀ nueva" if col in ("title_norm", "dish_image_path") else ""
    print(f"  {col:<45}  null: {n_null:>6,}  ({pct_null:.1%}){flag}")

Guardado en  : /home/ramon/DeepLearing_Recipes/df_clean_with_dish_images.parquet
Tamaño       : 102.5 MB
Shape        : (62126, 40)

Columnas del DataFrame final:
  recipe_title                                   null:      0  (0.0%)
  category                                       null:      0  (0.0%)
  subcategory                                    null:      0  (0.0%)
  description                                    null:      0  (0.0%)
  ingredients                                    null:      0  (0.0%)
  directions                                     null:      0  (0.0%)
  num_ingredients                                null:      0  (0.0%)
  num_steps                                      null:      0  (0.0%)
  ingredient_text                                null:      0  (0.0%)
  directions_text                                null:      0  (0.0%)
  combined_text                                  null:      0  (0.0%)
  ingredients_raw                                null:      0  (0.0

---
## Capa 2 — Imágenes de ingredientes

Fuente: `ingredients_canonical`. Matching: exact → fuzzy (`partial_ratio ≥ 72`). Caché por ingrediente único.

In [31]:
# ==========================================
# CARGA: parquet + ingredient_catalog
# ==========================================

# --- df_clean ---
try:
    # Verificar que tiene las columnas necesarias del paso anterior
    assert "ingredients_canonical" in df_clean.columns
    print(f"df_clean en memoria  : {df_clean.shape}")
except (NameError, AssertionError):
    print("Cargando df_clean_with_dish_images.parquet...")
    parquet_src = os.path.join(os.getcwd(), "df_clean_with_dish_images.parquet")
    df_clean = pd.read_parquet(parquet_src, engine="pyarrow")
    print(f"df_clean cargado     : {df_clean.shape}")

# --- ingredient_catalog ---
catalog_path = os.path.join(os.getcwd(), "ingredient_catalog.json")
with open(catalog_path, "r", encoding="utf-8") as f:
    ingredient_catalog = json.load(f)

print(f"ingredient_catalog   : {len(ingredient_catalog)} entradas")
print(f"\nMuestra de la columna ingredients_canonical:")
for v in df_clean["ingredients_canonical"].dropna().head(2):
    print(f"  {str(v)[:120]}")

df_clean en memoria  : (62126, 40)
ingredient_catalog   : 183 entradas

Muestra de la columna ingredients_canonical:
  ["ketchup", "beer", "worcestershire sauce", "onion powder", "cayenne", "baking potatoes", "olive oil cooking spray", "ga
  ["pound pork belly", "gochujang", "soy sauce", "honey", "ground ginger"]


### Funciones de matching

In [32]:
# ==========================================
# FUNCIONES DE MATCHING
# ==========================================
from rapidfuzz import process, fuzz

# Pre-calcular las keys del catálogo una sola vez (usadas en cada fuzzy search)
CATALOG_KEYS = list(ingredient_catalog.keys())

FUZZY_THRESHOLD = 72

def normalize_ingredient(name):
    name = str(name).lower().strip()
    name = re.sub(r'[^\w\s]', '', name)   # quitar puntuación
    name = re.sub(r'\s+', ' ', name).strip()
    name = re.sub(r's$', '', name)         # quitar plural simple
    return name

def match_ingredient(ingredient_name):
    norm = normalize_ingredient(ingredient_name)
    orig = str(ingredient_name).lower().strip()

    # 1 ── Exact match (normalizado)
    if norm in ingredient_catalog:
        return {
            "ingredient": ingredient_name,
            "image_path": ingredient_catalog[norm],
            "match_score": 100,
        }

    # 2 ── Exact match (sin normalizar)
    if orig in ingredient_catalog:
        return {
            "ingredient": ingredient_name,
            "image_path": ingredient_catalog[orig],
            "match_score": 100,
        }

    # 3 ── Fuzzy match
    result = process.extractOne(
        norm,
        CATALOG_KEYS,
        scorer=fuzz.partial_ratio,
        score_cutoff=FUZZY_THRESHOLD,
    )
    if result:
        matched_key, score, _ = result
        return {
            "ingredient": ingredient_name,
            "image_path": ingredient_catalog[matched_key],
            "match_score": int(score),
        }

    return {"ingredient": ingredient_name, "image_path": None, "match_score": 0}


# Smoke test
for test in ["garlic", "tomatoes", "baking potatoes", "olive oil", "shrimp", "butter"]:
    m = match_ingredient(test)
    img = m["image_path"].split("/")[-1] if m["image_path"] else "—"
    print(f"  {test:<25} → score={m['match_score']:3d}  img={img}")

  garlic                    → score=100  img=Image_10_jpg.rf.2e27e17d9ad6567dfb7e4bb33e22f66f.jpg
  tomatoes                  → score=100  img=0_100.jpg
  baking potatoes           → score= 93  img=Image_100_jpg.rf.94dafd52c5d9a6c014231c3a56c00a05.jpg
  olive oil                 → score=100  img=oil-in-bowl_1-jpg_jpg.rf.68c3b6aaba98d34729bbde5c3b7c33c8.jpg
  shrimp                    → score=  0  img=—
  butter                    → score=100  img=Butter_100-jpg_jpg.rf.33b10c2e8b7b6d7005c20683e17f723f.jpg


### Caché y aplicación

In [33]:
# ==========================================
# CACHÉ DE MATCHES
# ==========================================
from tqdm.auto import tqdm

# 1 ── Recolectar ingredientes canónicos únicos
unique_ingredients = set()
for text in df_clean["ingredients_canonical"].dropna():
    try:
        items = json.loads(text)
        for item in items:
            if isinstance(item, str) and item.strip():
                unique_ingredients.add(item.strip().lower())
    except (json.JSONDecodeError, TypeError):
        pass

print(f"Ingredientes canónicos únicos: {len(unique_ingredients):,}")

# 2 ── Construir caché  (fuzzy search una vez por ingrediente único)
match_cache = {}
for ingr in tqdm(sorted(unique_ingredients), desc="Cacheando matches"):
    match_cache[ingr] = match_ingredient(ingr)

n_cached_with_img = sum(1 for m in match_cache.values() if m["image_path"] is not None)
print(f"\nCaché listo: {len(match_cache):,} entradas")
print(f"Con imagen : {n_cached_with_img:,}  ({n_cached_with_img / len(match_cache):.1%})")

# ==========================================
# APLICAR A df_clean
# ==========================================

def get_recipe_ingredient_images(ingredients_canonical_str):
    if pd.isna(ingredients_canonical_str):
        return []
    try:
        items = json.loads(ingredients_canonical_str)
    except (json.JSONDecodeError, TypeError):
        return []
    result = []
    for item in items:
        if not isinstance(item, str) or not item.strip():
            continue
        key = item.strip().lower()
        # El caché cubre todos los ingredientes vistos en el dataset
        m = match_cache.get(key, {"ingredient": item, "image_path": None, "match_score": 0})
        result.append(m)
    return result

# Drop columna previa si existe (re-ejecución segura)
df_clean = df_clean.drop(columns=["ingredient_images"], errors="ignore")

tqdm.pandas(desc="Aplicando ingredient_images")
df_clean["ingredient_images"] = (
    df_clean["ingredients_canonical"].progress_apply(get_recipe_ingredient_images)
)

# Resumen
n_ingr_per_recipe = df_clean["ingredient_images"].apply(len)
print(f"\ningredient_images añadida:")
print(f"  Media de ingredientes/receta : {n_ingr_per_recipe.mean():.1f}")
print(f"  Rango                        : {n_ingr_per_recipe.min()} – {n_ingr_per_recipe.max()}")

Ingredientes canónicos únicos: 46,882


Cacheando matches:   0%|          | 0/46882 [00:00<?, ?it/s]


Caché listo: 46,882 entradas
Con imagen : 38,170  (81.4%)


Aplicando ingredient_images:   0%|          | 0/62126 [00:00<?, ?it/s]


ingredient_images añadida:
  Media de ingredientes/receta : 9.0
  Rango                        : 1 – 35


### Cobertura — Capa 2

In [34]:
# ==========================================
# REPORTE DE COBERTURA
# ==========================================
from itertools import chain

# Aplanar todas las listas de matches en un solo DataFrame
all_matches = list(chain.from_iterable(
    row or [] for row in df_clean["ingredient_images"]
))
df_flat = pd.DataFrame(all_matches)

print(f"Total apariciones de ingredientes: {len(df_flat):,}")

# Agregar por nombre de ingrediente
ingredient_coverage = (
    df_flat
    .groupby("ingredient", as_index=False)
    .agg(
        frequency   = ("ingredient",  "count"),
        has_image   = ("image_path",  lambda x: x.notna().any()),
        match_score = ("match_score", "max"),
    )
    .rename(columns={"ingredient": "ingredient_name"})
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)

# ── Tasa global ───────────────────────────────────────────────────────────────
total_apps   = len(df_flat)
covered_apps = int(df_flat["image_path"].notna().sum())
pct_covered  = covered_apps / total_apps if total_apps > 0 else 0

print(f"\n{'=' * 65}")
print(f"TASA GLOBAL DE COBERTURA")
print(f"{'=' * 65}")
print(f"  {pct_covered:.1%} de apariciones de ingredientes tienen imagen")
print(f"  ({covered_apps:,} de {total_apps:,} apariciones)")
n_with_img = ingredient_coverage["has_image"].sum()
n_without  = len(ingredient_coverage) - n_with_img
print(f"\n  Ingredientes únicos con imagen    : {n_with_img:,}")
print(f"  Ingredientes únicos sin imagen    : {n_without:,}")

# ── Top 20 más frecuentes SIN imagen ─────────────────────────────────────────
print(f"\nTop 20 ingredientes más frecuentes SIN imagen")
print(f"(candidatos para búsqueda manual o via API)")
top_missing = (
    ingredient_coverage[~ingredient_coverage["has_image"]]
    [["ingredient_name", "frequency"]]
    .head(20)
    .reset_index(drop=True)
)
display(top_missing)

# ── Top 10 CON imagen ─────────────────────────────────────────────────────────
print(f"\nTop 10 ingredientes más frecuentes CON imagen:")
top_with = (
    ingredient_coverage[ingredient_coverage["has_image"]]
    [["ingredient_name", "frequency", "match_score"]]
    .head(10)
    .reset_index(drop=True)
)
display(top_with)

Total apariciones de ingredientes: 560,115

TASA GLOBAL DE COBERTURA
  84.3% de apariciones de ingredientes tienen imagen
  (472,272 de 560,115 apariciones)

  Ingredientes únicos con imagen    : 38,170
  Ingredientes únicos sin imagen    : 8,712

Top 20 ingredientes más frecuentes SIN imagen
(candidatos para búsqueda manual o via API)


,ingredient_name,frequency
0,vanilla extract,7072
1,milk,3289
2,soy sauce,1649
3,worcestershire sauce,1498
4,½ milk,1456
5,½ vanilla extract,1452
6,pound ground beef,1221
7,mayonnaise,1140
8,shredded cheddar cheese,1119
9,dijon mustard,769



Top 10 ingredientes más frecuentes CON imagen:


,ingredient_name,frequency,match_score
0,white sugar,8646,100
1,salt,8368,100
2,all-purpose flour,7592,100
3,½ salt,6272,100
4,olive oil,6214,100
5,water,5317,100
6,large eggs,4899,85
7,butter,4638,100
8,baking powder,4182,100
9,eggs,3927,100


### Guardar archivos finales

- `df_clean_final.parquet`: corpus con `dish_image_path` e `ingredient_images`
- `ingredient_coverage.csv`: cobertura por ingrediente único

In [35]:
# ==========================================
# GUARDAR ARCHIVOS FINALES
# ==========================================

# ingredient_images se guarda como JSON string para máxima compatibilidad con parquet
df_save = df_clean.copy()
df_save["ingredient_images"] = df_save["ingredient_images"].apply(
    lambda lst: json.dumps(lst, ensure_ascii=False) if lst else "[]"
)

# --- df_clean_final.parquet ---
out_parquet = os.path.join(os.getcwd(), "df_clean_final.parquet")
df_save.to_parquet(out_parquet, index=False, engine="pyarrow")
size_mb = os.path.getsize(out_parquet) / 1_048_576
print(f"df_clean_final.parquet")
print(f"  Path   : {out_parquet}")
print(f"  Tamaño : {size_mb:.1f} MB")
print(f"  Shape  : {df_save.shape}")

# --- ingredient_coverage.csv ---
out_csv = os.path.join(os.getcwd(), "ingredient_coverage.csv")
ingredient_coverage.to_csv(out_csv, index=False, encoding="utf-8")
size_kb = os.path.getsize(out_csv) / 1024
print(f"\ningredient_coverage.csv")
print(f"  Path   : {out_csv}")
print(f"  Tamaño : {size_kb:.1f} KB")
print(f"  Rows   : {len(ingredient_coverage):,}")

# --- Resumen columnas finales ---
print(f"\nColumnas del DataFrame final ({len(df_save.columns)}):")
new_cols = {"title_norm", "dish_image_path", "ingredient_images"}
for col in df_save.columns:
    null_pct = df_save[col].isna().mean()
    mark = " ◀ nueva" if col in new_cols else ""
    print(f"  {col:<45}  null: {null_pct:.1%}{mark}")

df_clean_final.parquet
  Path   : /home/ramon/DeepLearing_Recipes/df_clean_final.parquet
  Tamaño : 115.7 MB
  Shape  : (62126, 41)

ingredient_coverage.csv
  Path   : /home/ramon/DeepLearing_Recipes/ingredient_coverage.csv
  Tamaño : 1893.7 KB
  Rows   : 46,882

Columnas del DataFrame final (41):
  recipe_title                                   null: 0.0%
  category                                       null: 0.0%
  subcategory                                    null: 0.0%
  description                                    null: 0.0%
  ingredients                                    null: 0.0%
  directions                                     null: 0.0%
  num_ingredients                                null: 0.0%
  num_steps                                      null: 0.0%
  ingredient_text                                null: 0.0%
  directions_text                                null: 0.0%
  combined_text                                  null: 0.0%
  ingredients_raw                        